In [1]:
import torch
print(torch.__version__)
import os
import re
from collections import Counter

2.10.0+cu126


In [2]:
with open("./the-verdict.txt", "r", encoding="utf-8") as file:
    text = file.read()

text[:200]

'I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no great surprise to me to hear that, in the height of his glory, he had dropped his painting, married a'

In [31]:
class BPETokenizer:
    def __init__(self):
        self.vocab = {}
        self.inverse_vocab = {}
        self.bpe_merges = {}

    def train(self, text, vocab_size, allowed_specials={"<|endoftext|>"}):
        unique_chars = [chr(i) for i in range(256)]
        unique_chars.extend(char for char in sorted(set(text)) if char not in unique_chars)
        self.vocab = {i:char for i, char in enumerate(unique_chars)}
        self.inverse_vocab = {char:i for i, char in enumerate(unique_chars)}

        if allowed_specials:
            for token in allowed_specials:
                if token not in self.inverse_vocab:
                    new_id = len(self.inverse_vocab)
                    self.vocab[new_id] = token
                    self.inverse_vocab[token] = new_id

        token_ids = [self.inverse_vocab[char] for char in text]

        for new_id in range(len(self.inverse_vocab), vocab_size):
            freq_pair = self.find_freq_pair(token_ids)
            if freq_pair is None:
                break
            token_ids = self.replace_pair(token_ids, freq_pair, new_id)
            self.bpe_merges[freq_pair] = new_id

        for (p0, p1), idx in self.bpe_merges.items():
            merged_token = self.vocab[p0] + self.vocab[p1]
            # print(f"Merged pair : {merged_token}, idx : {idx}")
            # print(f"p0 : {p0}, vocab[p0] : {self.vocab[p0]}, p1: {p1}, vocab[p1]: {self.vocab[p1]}")
            self.vocab[idx] = merged_token
            self.inverse_vocab[merged_token] = idx

    def find_freq_pair(self, token_ids:list):
        pairs = Counter(zip(token_ids, token_ids[1:]))

        if not pairs:
            return None
        
        return max(pairs.items(), key=lambda x: x[1])[0]
    
    def replace_pair(self, token_ids, freq_pair, idx):
        replaced = []
        i = 0
        while i  < (len(token_ids) -1):
            if (token_ids[i], token_ids[i+1]) == freq_pair:
                replaced.append(idx)
                i += 2
            else:
                replaced.append(token_ids[i])
                i += 1
        return replaced
    
    def tokenize_with_bpe(self, tokens):
        token_ids = [self.inverse_vocab.get(char, None) for char in tokens]

        if None in token_ids:
            missing_chars = [char for char, tid in zip(tokens, token_ids) if tid is None]
            raise ValueError(f"Characters not found in vocab : {missing_chars}")
        can_merge = True
        while can_merge and len(token_ids) > 1:
            can_merge = False
            new_tokens = []
            i = 0
            while i < len(token_ids)-1:
                pair = (token_ids[i], token_ids[i+1])
                if pair in self.bpe_merges:
                    merged_token_id = self.bpe_merges[pair]
                    new_tokens.append(merged_token_id)
                    i += 2
                    can_merge = True
                else:
                    new_tokens.append(token_ids[i])
                    i+= 1
            if  i < len(token_ids):
                new_tokens.append(token_ids[i])
            token_ids = new_tokens
            return token_ids
        
    def encode(self, text, allowed_specials=None):
        token_ids = []

        if allowed_specials is not None and len(allowed_specials) > 0:
            special_pattern = (
                "(" + "|".join(re.escape(tok) for tok in sorted(allowed_specials, key=len, reverse=True)) + ")"
            )
            last_index = 0
            for match in re.finditer(special_pattern, text):
                prefix = text[last_index:match.start()]
                token_ids.extexd(self.encode(prefix, allowed_specials=None))

                special_token = match.group(0)
                if special_token in self.inverse_vocab:
                    token_ids.append(self.inverse_vocab[special_token])
                else:
                    raise ValueError(f"Special token  {special_token} not found in vocabulary")
                last_index = match.end()
            
            text = text[last_index:]
            disallowed = [tok for tok in self.inverse_vocab
                      if tok.startswith("<|") and tok.endswith("|>") and tok not in allowed_specials]
        
            if disallowed:
                raise ValueError(f"Disallowed special tokens encountered in text : {disallowed}")
        
        tokens = []
        lines = text.split("\n")
        for i, line in enumerate(lines):
            if i > 0:
                tokens.append("\n")
            words = line.split()
            for j, word in enumerate(words):
                if j==0 and i > 0:
                    tokens.append(" " + word)
                elif j == 0:
                    tokens.append(word)
                else:
                    tokens.append(" " + word)

        for token in tokens:
            if token in self.inverse_vocab:
                token_ids.append(self.inverse_vocab[token])
            else:
                token_ids.extend(self.tokenize_with_bpe(token))
    
        return token_ids
    
    def decode(self, token_ids):
        decoded_string = ""
        for i, token_id in enumerate(token_ids):
            if token_id not in self.vocab:
                raise ValueError(f"Token id {token_id} not in vocabulary")
            token = self.vocab[token_id]
            if token == "\n":
                if decoded_string and not decoded_string.endswith(" "):
                    decoded_string += " "
                decoded_string += token
            else:
                decoded_string += token
        return decoded_string

In [32]:
tokenizer = BPETokenizer()
tokenizer.train(text, 400)

In [33]:
encoded_string = tokenizer.encode(text[:200])
decoded_string = tokenizer.decode(encoded_string)

decoded_string == text[:200]

True